In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (10.0, 6.0)

# Investment & Finance - Pairs Trading

<hr>

## 1. Mathematics: Integration, Cointegration, and Stationarity 

A commonly untested assumption in time series analysis is the stationarity of the data. **Data are stationary when the parameters of the data generating process (i.e mean, variance and standar deviation) do not change over time.** Many statistical tests, deep down in the fine print of their assumptions, require that the data being tested are stationary. 

As an example, let's consider two series, A and B. Series A is generated from a stationary process with fixed parameters, series B is generated with parameters that change over time.

#### - SERIES A: STATIONARY

In [ ]:
# Stationary (since it's coming from a normal distribution with mean=0 and std=1)
A = np.random.normal(0, 1, 100)
mean = A.mean()

plt.plot(A)
plt.hlines(mean, 0, len(A), linestyles='dashed', colors='r')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend(['Series A: Stationary', 'Mean'])
plt.show()

#### - SERIES B: ΝΟΝ-STATIONARY

In [ ]:
# Non-Stationary (since it's coming from a normal distribution but mean and std depend on time!)
B = pd.Series(index=range(100))

for t in range(100):
    B[t] = np.random.normal(t * 0.1, 1)
    
mean = B.mean()

plt.plot(B)
plt.hlines(mean, 0, len(B), linestyles='dashed', colors='r')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend(['Series B: Non-Stationary', 'Mean'])
plt.show()

As we can observe the computed mean will show the mean of all data points, but won't be useful for any forecasting of future state. It's meaningless when compared with any specfic time, as it's a collection of different states at different times mashed together. This is just a simple and clear example of why non-stationarity can screw with analysis, much more subtle problems can arise in practice.

#### - TESTING FOR STATIONARITY

Now we want to check for stationarity using a statistical test.

In [ ]:
# Define a function which returns a p-value to determine whether or not our time series is stationary!
# H_0 in adfuller is unit root exists (non-stationary)
# We must observe significant p-value to convince ourselves that the series is stationary

from statsmodels.tsa.stattools import adfuller

def check_for_stationarity(X, cutoff):
    
    pvalue = adfuller(X)[1]
    if pvalue < cutoff:
        print ('p-value = ' + str(pvalue) + ' (Stationary)')
    else:
        print ('p-value = ' + str(pvalue) + ' (Non-Stationary)')
    
check_for_stationarity(A, 0.01)
check_for_stationarity(B, 0.01)

#### - TESTING FOR $I(0)$

If we find that a series is stationary, then it must also be $I(0)$. Let's take our original stationary series A. Because A is stationary, we know it's also $I(0)$.

If one takes an $I(0)$ series and cumulatively sums it (discrete integration), the new series will be $I(1)$. By taking the cumlulative sum again, we obtain $I(2)$ . The same relation applies in general, to get $I(n)$ take an $I(0)$ series and iteratively take the cumulative sum $n$ times.

In [ ]:
A1 = np.cumsum(A)
A2 = np.cumsum(A1)

plt.figure(figsize=(18,8))

plt.subplot(1,3,1)
plt.plot(A)
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('I(0)')
plt.legend(['Series A'])

plt.subplot(1,3,2)
plt.plot(A1)
plt.xlabel('Time')
plt.title('I(1)')
plt.legend(['Series A1'])
          
plt.subplot(1,3,3)          
plt.plot(A2)
plt.xlabel('Time')
plt.title('I(2)')
plt.legend(['Series A2']);

plt.show()    

Conversely, to find the order of integration of a given series, we perform the inverse of a cumulative sum, which is the $\Delta$ or itemwise difference function. Specifically

$$(1-L) X_t = X_t - X_{t-1} = \Delta X$$

$$(1-L)^d X_t$$

In this case $L$ is the lag operator. Sometimes also written as $B$ for 'backshift'. $L$ fetches the second to last elements in a time series, and $L^k$ fetches the k-th to last elements. So 

$$L X_t = X_{t-1}$$

and

$$(1-L) X_t = X_t - X_{t-1}$$

A series $Y_t$ is $I(1)$ if the $Y_t - Y_t-1$ is $I(0)$. In other words, if you take an $I(0)$ series and cumulatively sum it, you should get an $I(1)$ series.


###Important Take-Away

Once all the math has settled, remember that any stationary series is $I(0)$

Hence, we can take series B, that we know is not stationary, and take the differences in order to obtain a stationary state.

In [ ]:
check_for_stationarity(B, 0.01)

B1 = B.diff()[1:]
check_for_stationarity(B1, 0.01)

#### - COINTEGRATION

- **Linear Combination**

A linear combination of the time series ($X_1$, $X_2$, $\dots$, $X_k$) is a new time series $Y$ constructed as follows for any set of real numbers $b_1 \dots b_k$

$$Y = b_1X_1 + b_2X_2 + \dots + b_kX_k$$

- **Cointegration**

For some set of time series ($X_1$, $X_2$, $\dots$, $X_k$), if all series are $I(1)$, and some linear combination of them is $I(0)$, we say the set of time series is cointegrated.

Example:

$X_1$, $X_2$, and $X_3$ are all $I(1)$, and $2X_1 + X_2 + 0X_3 = 2X_1 + X_2$ is $I(0)$. In this case the time series are cointegrated.

The intuition here is that for some linear combination of the series, the result lacks much auto-covariance and is mostly noise. This is useful for cases such as pairs trading, in which we find two assets whose prices are cointegrated. Since the linear combination of their prices $b_1A_1 + b_2A_2$ is noise, we can bet on the relationship $b_1A_1 + b_2A_2$ mean reverting and place trades accordingly.

Let's make some data to demonstrate this.

In [ ]:
# Generate a stationary random X1 and integrate it to make it I(1)
X1 = np.random.normal(0, 1, 100).cumsum()

# Make an X2 that is X1 plus some noise
X2 = X1 + np.random.normal(0, 1, 100)

plt.plot(X1)
plt.plot(X2)
plt.xlabel('Time')
plt.ylabel('Series Value')
plt.legend(['X1','X2'])
plt.show()

In [ ]:
# X1 is not stationary since it's I(1).
check_for_stationarity(X1,0.01)

# Since X2 is just an I(1) series plus some stationary noise, it should still be $I(1)$. Let's check this.
check_for_stationarity(X2,0.01)

# But the cointegration X2 - X1 is stationary.
check_for_stationarity((X2-X1), 0.01)

Of course the problem is to find the parameter of the linear combination.

In practice a common way to do this for pairs of time series is to use linear regression to estimate $\beta$ in the following model.

$$X_2 = \alpha + \beta X_1 + \epsilon$$

The idea is that if the two are cointegrated we can remove $X_2$'s depedency on $X_1$, leaving behind stationary noise. The combination $X_2 - \beta X_1 = \alpha + \epsilon$ should be stationary.

Hence, for the previous X1 and X2 we can fit a linear regression and obtain $\beta$. Then $X_2 - \beta X_1$ should be stationary.

In [ ]:
b = stats.linregress(X2,X1)[0]
check_for_stationarity((X2-b*X1),0.01)

This is only a forecast!

Remember as with anything else, you should not assume that because some set of assets have passed a cointegration test historically, they will continue to remain cointegrated. You need to verify that consistent behavior occurs, and use various model validation techniques as you would with any model.

Luckily there are some pre-built tests for cointegration.

In [ ]:
from statsmodels.tsa.stattools import coint

coint(X1,X2)

<hr>

## 2. Pairs Trading

Pairs trading is a classic example of a strategy based on mathematical analysis. The principle is as follows. Let's say you have a pair of securities X and Y that have some underlying economic link. An example might be two companies that manufacture the same product, or two companies in one supply chain. If we can model this economic link with a mathematical model, we can make trades on it.

#### - Hedging

Because you'd like to protect yourself from bad markets, often times short sales will be used to hedge long investments. Because a short sale makes money if the security sold loses value, and a long purchase will make money if a security gains value, one can long parts of the market and short others. That way if the entire market falls off a cliff, we'll still make money on the shorted securities and hopefully break even. In the case of two securities we'll call it a hedged position when we are long on one security and short on the other.

#### - The Trick: Where it all comes together

Because the securities drift towards and apart from each other, there will be times when the distance is high and times when the distance is low. The trick of pairs trading comes from maintaining a hedged position across X and Y. If both securities go down, we neither make nor lose money, and likewise if both go up. We make money on the spread of the two reverting to the mean. In order to do this we'll watch for when X and Y are far apart, then short Y and long X. Similarly we'll watch for when they're close together, and long Y and short X.

#### - Going Long the Spread

This is when the spread is small and we expect it to become larger. We place a bet on this by longing Y and shorting X.

#### - Going Short the Spread

This is when the spread is large and we expect it to become smaller. We place a bet on this by shorting Y and longing X.

#### - Specific Bets

One important concept here is that we are placing a bet on one specific thing, and trying to reduce our bet's dependency on other factors such as the market.

#### - Finding real securities that behave like this

The best way to do this is to start with securities you suspect may be cointegrated and perform a statistical test. If you just run statistical tests over all pairs, you'll fall prey to multiple comparison bias.

Here's a method to look through a list of securities and test for cointegration between all pairs. It returns a cointegration test score matrix, a p-value matrix, and any pairs for which the p-value was less than $0.05$.

#### -  WARNING: This will incur a large amount of multiple comparisons bias.
The methods for finding viable pairs all live on a spectrum. At one end there is the formation of an economic hypothesis for an individual pair. You have some extra knowledge about an economic link that leads you to believe that the pair is cointegrated, so you go out and test for the presence of cointegration. In this case you will incur no multiple comparisons bias. At the other end of the spectrum, you perform a search through hundreds of different securities for any viable pairs according to your test. In this case you will incur a very large amount of multiple comparisons bias. 

Multiple comparisons bias is the increased chance to incorrectly generate a significant p-value when many tests are run. If 100 tests are run on random data, we should expect to see 5 p-values below $0.05$ on expectation. Because we will perform $n(n-1)/2$ comparisons, we should expect to see many incorrectly significant p-values. For the sake of example will will ignore this and continue. In practice a second verification step would be needed if looking for pairs this way. Another approach is to pick a small number of pairs you have reason to suspect might be cointegrated and test each individually. This will result in less exposure to multiple comparisons bias.

Now lets start deploying a toy model.

#### - Generating Two Fake Securities

We model X's daily returns by drawing from a normal distribution. Then we perform a cumulative sum to get the value of X on each day. That's stock_1. For stock_2 remember that it is supposed to have a deep economic link to stock_1, so the price of stock_2 should vary pretty similarly. We model this by taking stock_1, shifting it up and adding some random noise drawn from a normal distribution.

In [ ]:
stock_1 = pd.Series(np.cumsum(np.random.normal(0, 1, 100))) + 50
stock_2 = stock_1 + 5 + np.random.normal(0, 1, 100)

plt.plot(stock_1)
plt.plot(stock_2)
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend(['Stock 1','Stock 2'])
plt.show()

We've constructed an example of two cointegrated series. Cointegration is a more subtle relationship than correlation. If two time series are cointegrated, there is some linear combination between them that will vary around a mean. At all points in time, the combination between them is related to the same probability distribution. 

We'll plot the difference between the two now so we can see how this looks.

In [ ]:
# Plot the spread
(stock_2 - stock_1).plot() 
plt.axhline((stock_2 - stock_1).mean(), color='red', linestyle='--') # Add the mean
plt.xlabel('Time')
plt.legend(['Price Spread', 'Mean'])
plt.show()

Finally we can check the cointegration.

In [ ]:
# Cointegration checking
check_for_stationarity((stock_2 - stock_1),0.01)

# Or using the built in function
score, pvalue, _ = coint(stock_1,stock_2)

# And finaly we use the spread defined with the beta from linear regression
b = (stats.linregress(stock_2,stock_1)[0])
spread = stock_2 - b * stock_1

spread.plot()
plt.axhline(spread.mean(), color='red', linestyle='--')
plt.axhline(spread.mean() + spread.std(), color='green', linestyle='--')
plt.axhline(spread.mean() - spread.std(), color='green', linestyle='--')
plt.legend(['Price Spread', 'Mean', 'Std'])
plt.show()

#### - Simple Strategy:
* Go "Long" the spread whenever the spread is below mean - std
* Go "Short" the spread when the spread is above mean + std
* Exit positions when the spread approaches zero

#### - Trading using constantly updating statistics

In general taking a statistic over your whole sample size can be bad. For example, if the market is moving up, and both securities with it, then your average price over the last 3 years may not be representative of today. For this reason traders often use statistics that rely on rolling windows of the most recent data.

The problem here is that we calculated a beta based on the whole dataset. We need to use a rolling beta, a rolling estimate of how our spread should be calculated, in order to keep all of our parameters up to date. In general a moving average is just an average over the last $n$ datapoints for each given time. It will be undefined for the first $n$ datapoints in our series. Shorter moving averages will be more jumpy and less reliable, but respond to new information quickly. Longer moving averages will be smoother, but take more time to incorporate new information.

In [ ]:
# Get the spread between the 2 stocks
# Calculate rolling beta coefficient
rolling_beta = pd.ols(y=stock_1, x=stock_2, window_type='rolling', window=30)
spread = stock_2 - rolling_beta.beta['x'] * stock_1
spread.name = 'spread'

# Get the 1 day moving average of the price spread
spread_mavg1 = spread.rolling(window=1).mean()
spread_mavg1.name = 'spread 1d mavg'

# Get the 30 day moving average
spread_mavg30 = spread.rolling(window=30).mean()
spread_mavg30.name = 'spread 30d mavg'

plt.plot(spread_mavg1.index, spread_mavg1.values)
plt.plot(spread_mavg30.index, spread_mavg30.values)

plt.legend(['1 Day Spread MAVG', '30 Day Spread MAVG'])

plt.ylabel('Spread')
plt.show()

Finally, here is a useful function that calculates cointegrations between stocks given their prices!

In [ ]:
# Gets returns of stocks, finds cointegrated stocks, and prints a heatmap of them
def find_cointegrated_pairs(data):
    n = data.shape[1]
    score_matrix = np.zeros((n, n))
    pvalue_matrix = np.ones((n, n))
    keys = data.keys()
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            S1 = data[keys[i]]
            S2 = data[keys[j]]
            result = coint(S1, S2)
            score = result[0]
            pvalue = result[1]
            score_matrix[i, j] = score
            pvalue_matrix[i, j] = pvalue
            if pvalue < 0.05:
                pairs.append((keys[i], keys[j]))
    
    sns.heatmap(pvalues, xticklabels=data.columns, yticklabels=data.columns, cmap='RdYlGn_r' , mask = (pvalues >= 0.05))
    plt.show()
    return score_matrix, pvalue_matrix, pairs